In [1]:
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt
from itertools import combinations
from statsmodels.tsa.stattools import coint
from scipy.stats import ttest_rel

pd.set_option("display.width", 140)

In [2]:
UNIVERSE = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "LINK", "AVAX", "DOT", 
            "LTC", "ATOM", "AAVE", "UNI", "ARB", "OP", "SUI", "NEAR", "ALGO", "FIL", 
            "MATIC", "TRX", "BCH", "ETC", "XLM", "ICP", "APT", "SEI", "INJ", "RENDER", 
            "FET", "TIA", "HBAR", "VET", "THETA", "EOS", "SAND", "MANA", "GRT", "MKR"]
SYMBOLS = [f"{coin}/USD" for coin in UNIVERSE]

In [3]:
exchange = ccxt.coinbase()
markets = exchange.load_markets()

symbols = [f"{coin}/USD" for coin in UNIVERSE if f"{coin}/USD" in markets]
pairs = list(combinations(symbols, 2))
print(f"{len(symbols)} coins available, {len(pairs)} candidate pairs")

37 coins available, 666 candidate pairs


In [4]:
def to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close", "volume"]].copy()
    df = df.rename(columns={"close": symbol, "volume": f"{symbol}_volume"})
    return df

In [5]:
exchange = ccxt.coinbase({
    "enableRateLimit": True,
    "timeout": 30000,  
})

def fetch_ohlcv_history(exchange, symbol, timeframe, since_ms, limit=300, max_retries=5):
    all_rows = []
    since = since_ms
    now_ms = exchange.milliseconds()

    while since < now_ms:
        batch = None
        for attempt in range(max_retries):
            try:
                batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
                break
            except (ccxt.RequestTimeout, ccxt.NetworkError) as e:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

        if not batch:
            break

        all_rows.extend(batch)
        since = batch[-1][0] + 1

    return all_rows

In [ ]:
YEARS_OF_HISTORY = 5 
TIMEFRAME = "1h"

since_dt = pd.Timestamp.utcnow() - pd.DateOffset(years=YEARS_OF_HISTORY)
HISTORY_SINCE = exchange.parse8601(since_dt.strftime("%Y-%m-%dT%H:%M:%SZ"))

prices = None
for coin in UNIVERSE:
    symbol = f"{coin}/USD"
    if symbol not in markets:
        continue
    data = fetch_ohlcv_history(exchange, symbol, timeframe=TIMEFRAME, since_ms=HISTORY_SINCE)
    if not data:
        continue
    df = to_df(data, coin)
    prices = df if prices is None else prices.merge(df, on="timestamp", how="inner")

prices = prices.sort_values("timestamp").reset_index(drop=True)
print(f"{len(prices)} aligned hourly bars, {prices['timestamp'].min()} -> {prices['timestamp'].max()}")

In [ ]:
TIMEFRAME_HOURS = [4, 8, 12]
TRAIN_FRAC = 0.70
FIT_FRAC_OF_TRAIN = 0.75   # inner train = 52.5% of total, validation = 17.5%

FEE_BPS_MAIN = 20.0
Z_ENTRY = 2.5
Z_EXIT = 0.5
BB_K = 1.5
MIN_PERSIST = 2
COOLDOWN_BARS_HOURS = 48
VOL_WINDOW_HOURS = 14 * 24
VOL_LOW_Z = -0.75
VOL_HIGH_Z = 1.0
VOL_BLOCK_Z = 1.75
LOW_VOL_MULT = 1.25
MID_VOL_MULT = 1.00
HIGH_VOL_MULT = 0.50

MIN_HALF_LIFE_HOURS = 12
MAX_HALF_LIFE_HOURS = 20 * 24
MAX_SIGNAL_WINDOW_HOURS = 20 * 24
MIN_BETA_STABILITY_CV = 0.25
MIN_CORR = 0.80
MAX_EG_PVALUE = 0.10
MIN_CONVERGENCE_EVENTS = 3
MAX_SELECTED_PAIRS = 8

In [ ]:
prices_1h = prices.copy().sort_values("timestamp").reset_index(drop=True)

In [ ]:
def align_pair_prices(prices, asset_a, asset_b):
    cols = ["timestamp", asset_a, asset_b]
    vol_cols = [f"{asset_a}_volume", f"{asset_b}_volume"]
    needed = cols + [c for c in vol_cols if c in prices.columns]
    out = prices[needed].dropna().copy()
    out = out.sort_values("timestamp").reset_index(drop=True)
    return out

In [ ]:
def estimate_ols_spread(df, asset_a, asset_b):
    """
    Fit the hedge ratio via OLS (used ONLY on the train set) and run the
    Engle-Granger cointegration test on the two log-price series.
    Two nonstationary series can produce a highly significant OLS slope
    just because both trend -- that p-value is NOT a cointegration test,
    so we report the Engle-Granger p-value instead.
    """
    out = df.copy()
    out[f"log_{asset_a}"] = np.log(out[asset_a])
    out[f"log_{asset_b}"] = np.log(out[asset_b])

    X = sm.add_constant(out[f"log_{asset_b}"])
    y = out[f"log_{asset_a}"]
    model = sm.OLS(y, X).fit()
    
    alpha = model.params["const"]
    beta = model.params[f"log_{asset_b}"]
    corr = out[[f"log_{asset_a}", f"log_{asset_b}"]].corr().iloc[0, 1]
    out["spread"] = model.resid

    eg_stat, eg_pvalue, eg_crit = coint(out[f"log_{asset_a}"], out[f"log_{asset_b}"])

    return out, alpha, beta, corr, eg_pvalue

In [ ]:
def apply_fixed_spread(df, asset_a, asset_b, alpha, beta):
    """
    Build the spread on a NEW sample (e.g. the test set) using an alpha/beta
    that was already estimated elsewhere (the train set). No refitting here,
    so no look-ahead when this is used out-of-sample.
    """
    out = df.copy()
    out[f"log_{asset_a}"] = np.log(out[asset_a])
    out[f"log_{asset_b}"] = np.log(out[asset_b])
    out["spread"] = out[f"log_{asset_a}"] - (alpha + beta * out[f"log_{asset_b}"])
    return out

In [ ]:
def spread_half_life(spread):
    s = pd.Series(spread).dropna()
    lag = s.shift(1)
    diff = s - lag
    tmp = pd.concat([diff, lag], axis=1).dropna()
    tmp.columns = ["diff", "lag"]
    if len(tmp) < 5:
        return np.nan
    model = sm.OLS(tmp["diff"], sm.add_constant(tmp["lag"])).fit()
    lam = model.params["lag"]
    if lam >= 0:
        return np.nan
    return -np.log(2) / lam

In [ ]:
def spread_crossings(z, threshold=2.0):
    z = pd.Series(z).dropna()
    if len(z) < 2:
        return 0
    upper = ((z.shift(1) < threshold) & (z >= threshold)).sum()
    lower = ((z.shift(1) > -threshold) & (z <= -threshold)).sum()
    return int(upper + lower)

In [ ]:
def pair_summary(prices, asset_a, asset_b, crossing_z=2.0):
    df, alpha, beta, corr, eg_pvalue = estimate_ols_spread(prices, asset_a, asset_b)

    spread = df["spread"].dropna()
    spread_vol = spread.std()
    hl = spread_half_life(spread)

    z = (spread - spread.mean()) / spread.std()
    crossings = spread_crossings(z, threshold=crossing_z)

    return pd.Series({
        "Pair": f"{asset_a}/{asset_b}",
        "Asset_A": asset_a,
        "Asset_B": asset_b,
        "Corr": corr,
        "EG_Coint_Pvalue": eg_pvalue, 
        "Alpha": alpha,
        "Beta": beta,
        "Spread Vol": spread_vol,
        "Half_life": hl,
        "Z_score Crossings": crossings
    })

In [ ]:
def rolling_zscore(series, window=25):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std

In [ ]:
def build_zscore_positions(z, entry_z=2.0, exit_z=0.25, min_persist=2, cooldown_bars=0, allow_entry=None):
    position = 0
    cooldown = 0
    below_count = 0
    above_count = 0
    positions = []

    for i, val in enumerate(z):
        if pd.isna(val):
            positions.append(0)
            continue

        if val < -entry_z:
            below_count += 1
        else:
            below_count = 0

        if val > entry_z:
            above_count += 1
        else:
            above_count = 0

        if cooldown > 0:
            cooldown -= 1

        entry_ok = True if allow_entry is None else bool(allow_entry.iloc[i])

        if position == 0:
            if cooldown == 0 and entry_ok and below_count >= min_persist:
                position = 1
            elif cooldown == 0 and entry_ok and above_count >= min_persist:
                position = -1

        elif position == 1:
            if val > -exit_z:
                position = 0
                cooldown = cooldown_bars

        elif position == -1:
            if val < exit_z:
                position = 0
                cooldown = cooldown_bars

        positions.append(position)

    return pd.Series(positions, index=z.index)

In [ ]:
def build_bollinger_positions(spread, window=25, band_k=1.0, min_persist=2, cooldown_bars=0, allow_entry=None):
    mid = spread.rolling(window).mean()
    std = spread.rolling(window).std()
    upper = mid + band_k * std
    lower = mid - band_k * std

    position = 0
    cooldown = 0
    below_count = 0
    above_count = 0
    positions = []

    for i, (s, m, u, l) in enumerate(zip(spread, mid, upper, lower)):
        if pd.isna(s) or pd.isna(m) or pd.isna(u) or pd.isna(l):
            positions.append(0)
            continue

        if s < l:
            below_count += 1
        else:
            below_count = 0

        if s > u:
            above_count += 1
        else:
            above_count = 0

        if cooldown > 0:
            cooldown -= 1

        entry_ok = True if allow_entry is None else bool(allow_entry.iloc[i])

        if position == 0:
            if cooldown == 0 and entry_ok and below_count >= min_persist:
                position = 1
            elif cooldown == 0 and entry_ok and above_count >= min_persist:
                position = -1

        elif position == 1:
            if s >= m:
                position = 0
                cooldown = cooldown_bars

        elif position == -1:
            if s <= m:
                position = 0
                cooldown = cooldown_bars

        positions.append(position)

    out = pd.Series(positions, index=spread.index)
    return out, mid, upper, lower

In [ ]:
def resample_price_volume(df_1h, bar_hours):
    """Resample price with last and volume with SUM, preserving the timestamps."""
    x = df_1h.copy()
    x["timestamp"] = pd.to_datetime(x["timestamp"])
    x = x.set_index("timestamp").sort_index()

    price_cols = [c for c in x.columns if not c.endswith("_volume")]
    volume_cols = [c for c in x.columns if c.endswith("_volume")]

    price_part = x[price_cols].resample(f"{bar_hours}h").last()
    volume_part = x[volume_cols].resample(f"{bar_hours}h").sum()

    out = pd.concat([price_part, volume_part], axis=1).dropna(how="all")
    out = out.reset_index()
    return out

In [ ]:
def split_three_periods(df, train_frac=TRAIN_FRAC, fit_frac_of_train=FIT_FRAC_OF_TRAIN):
    n = len(df)
    train_end = int(n * train_frac)
    fit_end = int(train_end * fit_frac_of_train)

    fit = df.iloc[:fit_end].reset_index(drop=True)
    val = df.iloc[fit_end:train_end].reset_index(drop=True)
    test = df.iloc[train_end:].reset_index(drop=True)

    return fit, val, test

In [ ]:
def bars_from_hours(hours, bar_hours, minimum=5, maximum=None):
    bars = max(minimum, int(round(float(hours) / bar_hours)))
    if maximum is not None:
        bars = min(bars, maximum)
    return bars

def hours_from_bars(bars, bar_hours):
    return float(bars * bar_hours)

In [ ]:
DATASETS = {}
for tf_hours in TIMEFRAME_HOURS:
    resampled = resample_price_volume(prices_1h, tf_hours)
    fit, val, test = split_three_periods(resampled)
    DATASETS[tf_hours] = {
        "all": resampled,
        "fit": fit,
        "val": val,
        "test": test,
    }
    print(
        f"{tf_hours}h: fit={len(fit)} bars, val={len(val)} bars, test={len(test)} bars; "
        f"range={resampled['timestamp'].min()} -> {resampled['timestamp'].max()}"
    )

In [ ]:
def rolling_beta_stability(df, asset_a, asset_b, n_blocks=4):
    """Estimate OLS hedge ratio separately in several chronological blocks.

    The coefficient of variation of those beta estimates is a simple, interpretable
    hedge-ratio stability measure.  It uses only the supplied training sample.
    """
    x = df[[asset_a, asset_b]].dropna().copy()
    if len(x) < 40:
        return np.nan, np.nan, []

    x[f"log_{asset_a}"] = np.log(x[asset_a])
    x[f"log_{asset_b}"] = np.log(x[asset_b])

    edges = np.linspace(0, len(x), n_blocks + 1).astype(int)
    betas = []

    for i in range(n_blocks):
        seg = x.iloc[edges[i]:edges[i + 1]]
        if len(seg) < 20:
            continue
        X = sm.add_constant(seg[f"log_{asset_b}"])
        y = seg[f"log_{asset_a}"]
        try:
            fit = sm.OLS(y, X).fit()
            betas.append(float(fit.params[f"log_{asset_b}"]))
        except Exception:
            continue

    if len(betas) < 2:
        return np.nan, np.nan, betas

    beta_mean = np.mean(betas)
    beta_cv = np.std(betas, ddof=1) / max(abs(beta_mean), 1e-8)
    beta_range = np.max(betas) - np.min(betas)
    return float(beta_cv), float(beta_range), betas


In [ ]:
def count_convergence_events(z, entry=2.0, exit=0.5):
    """Count transitions from a large deviation back inside a near-mean zone."""
    z = pd.Series(z).dropna()
    outside = z.abs() >= entry
    inside = z.abs() <= exit
    return int((outside.shift(1, fill_value=False) & inside).sum())

In [ ]:
def generic_screen_backtest(
    df, asset_a, asset_b, alpha, beta, half_life_hours, bar_hours,
    fee_bps=FEE_BPS_MAIN
):
    """A deliberately simple z-score reversal used only for pair screening.

    It is NOT the final optimized strategy.  Its purpose is to ask:
    'Was this pair historically reversible enough to survive estimated costs?'
    """
    out = apply_fixed_spread(df, asset_a, asset_b, alpha, beta)

    window_hours = np.clip(2.0 * half_life_hours, 24.0, MAX_SIGNAL_WINDOW_HOURS)
    window = bars_from_hours(window_hours, bar_hours, minimum=5)
    z = rolling_zscore(out["spread"], window=window)

    pos = build_zscore_positions(
        z,
        entry_z=Z_ENTRY,
        exit_z=Z_EXIT,
        min_persist=MIN_PERSIST,
        cooldown_bars=bars_from_hours(COOLDOWN_BARS_HOURS, bar_hours, minimum=0),
    )

    gross = 1.0 + abs(beta)
    w_a = pos / gross
    w_b = -beta * pos / gross

    ret_a = out[asset_a].pct_change()
    ret_b = out[asset_b].pct_change()

    raw_ret = w_a.shift(1) * ret_a + w_b.shift(1) * ret_b
    turnover = w_a.diff().abs().fillna(0) + w_b.diff().abs().fillna(0)
    cost = turnover * fee_bps / 10000.0
    net_ret = raw_ret - cost

    r = net_ret.dropna()
    ann = np.sqrt((24.0 * 365.0) / bar_hours)
    vol = r.std()
    sharpe = np.nan if vol == 0 or np.isnan(vol) else (r.mean() / vol) * ann

    return {
        "ScreenWindowHours": window * bar_hours,
        "ScreenSharpe": float(sharpe) if not np.isnan(sharpe) else np.nan,
        "ScreenTurnover": float(turnover.mean()),
        "ScreenCumReturn": float((1.0 + r).prod() - 1.0) if len(r) else np.nan,
    }


In [ ]:
def pair_training_features(df, asset_a, asset_b, bar_hours, fee_bps=FEE_BPS_MAIN):
    pair_df, alpha, beta, corr, eg_pvalue = estimate_ols_spread(df, asset_a, asset_b)
    spread = pair_df["spread"].dropna()

    if len(spread) < 50:
        return None

    half_life_bars = spread_half_life(spread)
    half_life_hours = np.nan if np.isnan(half_life_bars) else half_life_bars * bar_hours

    spread_std = float(spread.std())
    z_full = (spread - spread.mean()) / max(spread.std(), 1e-12)
    convergence = count_convergence_events(z_full, entry=2.0, exit=0.5)

    beta_cv, beta_range, beta_path = rolling_beta_stability(pair_df, asset_a, asset_b)

    screen = generic_screen_backtest(
        df, asset_a, asset_b, alpha, beta,
        half_life_hours if not np.isnan(half_life_hours) else MAX_HALF_LIFE_HOURS,
        bar_hours,
        fee_bps=fee_bps,
    )

    days = max((df["timestamp"].iloc[-1] - df["timestamp"].iloc[0]).total_seconds() / 86400.0, 1e-9)
    convergence_per_100d = convergence / days * 100.0

    return {
        "Pair": f"{asset_a}/{asset_b}",
        "Asset_A": asset_a,
        "Asset_B": asset_b,
        "Corr": corr,
        "EG_Coint_Pvalue": eg_pvalue,
        "Alpha": alpha,
        "Beta": beta,
        "SpreadVol": spread_std,
        "HalfLifeHours": half_life_hours,
        "BetaCV": beta_cv,
        "BetaRange": beta_range,
        "ConvergenceEvents": convergence,
        "ConvergencePer100d": convergence_per_100d,
        **screen,
    }

In [ ]:
def normalize_rank(s, higher_is_better=True):
    s = pd.Series(s, index=s.index, dtype=float)
    rank = s.rank(pct=True, method="average")
    return rank if higher_is_better else (1.0 - rank + 1.0 / len(rank))

In [ ]:
def select_pairs_from_training(df, bar_hours, fee_bps=FEE_BPS_MAIN):
    assets = [c for c in UNIVERSE if c in df.columns]
    pair_list = list(combinations(assets, 2))

    rows = []
    for a, b in pair_list:
        feat = pair_training_features(df, a, b, bar_hours, fee_bps=fee_bps)
        if feat is not None:
            rows.append(feat)

    table = pd.DataFrame(rows)
    if table.empty:
        raise ValueError("No pair diagnostics could be computed on the training period.")

    # Core statistical gates.  The other criteria are deliberately used as
    # ranking dimensions rather than brittle hard gates, because a particular
    # timeframe can otherwise end up with zero eligible pairs.
    eligible = table.copy()
    eligible = eligible[eligible["Corr"] >= MIN_CORR]
    eligible = eligible[eligible["EG_Coint_Pvalue"] <= MAX_EG_PVALUE]
    eligible = eligible[eligible["HalfLifeHours"].between(MIN_HALF_LIFE_HOURS, MAX_HALF_LIFE_HOURS)]

    if eligible.empty:
        raise ValueError("No pairs passed the core correlation / cointegration / half-life filters.")

    # Prefer positive net training Sharpe when the universe gives us enough
    # such pairs, but do not make the whole pipeline fail when a timeframe is
    # short on profitable pairs.
    positive_sharpe = eligible[eligible["ScreenSharpe"] > 0]
    if len(positive_sharpe) >= min(2, MAX_SELECTED_PAIRS):
        eligible = positive_sharpe.copy()

    eligible = eligible.copy()
    eligible["ScreenSharpe"] = eligible["ScreenSharpe"].replace([np.inf, -np.inf], np.nan).fillna(-5.0)
    eligible["ScreenTurnover"] = eligible["ScreenTurnover"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    eligible["ConvergencePer100d"] = eligible["ConvergencePer100d"].fillna(0.0)
    eligible["BetaCV"] = eligible["BetaCV"].fillna(10.0)
    eligible["Score_Coint"] = normalize_rank(eligible["EG_Coint_Pvalue"], higher_is_better=False)
    eligible["Score_Corr"] = normalize_rank(eligible["Corr"], higher_is_better=True)
    eligible["Score_BetaStability"] = normalize_rank(eligible["BetaCV"].fillna(eligible["BetaCV"].median()), higher_is_better=False)
    eligible["Score_HalfLife"] = normalize_rank(eligible["HalfLifeHours"], higher_is_better=False)
    eligible["Score_SpreadVol"] = normalize_rank(eligible["SpreadVol"], higher_is_better=False)
    eligible["Score_Convergence"] = normalize_rank(eligible["ConvergencePer100d"], higher_is_better=True)
    eligible["Score_NetSharpe"] = normalize_rank(eligible["ScreenSharpe"], higher_is_better=True)
    eligible["Score_Turnover"] = normalize_rank(eligible["ScreenTurnover"], higher_is_better=False)

    # Stability, convergence, net Sharpe and turnover are therefore part of
    # selection even though they are not all hard cutoffs.
    eligible["SelectionScore"] = (
        0.15 * eligible["Score_Coint"] +
        0.10 * eligible["Score_Corr"] +
        0.15 * eligible["Score_BetaStability"] +
        0.10 * eligible["Score_HalfLife"] +
        0.10 * eligible["Score_SpreadVol"] +
        0.10 * eligible["Score_Convergence"] +
        0.20 * eligible["Score_NetSharpe"] +
        0.10 * eligible["Score_Turnover"]
    )

    selected = eligible.sort_values("SelectionScore", ascending=False).head(MAX_SELECTED_PAIRS).reset_index(drop=True)
    return selected, table


In [ ]:
def rolling_zscore(series, window):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std.replace(0, np.nan)


def volume_regime(df, asset_a, asset_b, bar_hours):
    """Return a volume-based exposure multiplier and entry permission.

    We do NOT add the raw volumes of two different coins because their units
    are not comparable.  Each volume is first normalized by its own rolling
    median, then the two relative-volume measures are averaged.
    """
    out = pd.DataFrame(index=df.index)
    vol_window = bars_from_hours(VOL_WINDOW_HOURS, bar_hours, minimum=5)

    va = df[f"{asset_a}_volume"] if f"{asset_a}_volume" in df.columns else pd.Series(1.0, index=df.index)
    vb = df[f"{asset_b}_volume"] if f"{asset_b}_volume" in df.columns else pd.Series(1.0, index=df.index)

    ma = va.rolling(vol_window, min_periods=max(3, vol_window // 4)).median()
    mb = vb.rolling(vol_window, min_periods=max(3, vol_window // 4)).median()

    rel_a = va / ma.replace(0, np.nan)
    rel_b = vb / mb.replace(0, np.nan)
    relative_volume = 0.5 * (rel_a + rel_b)

    log_rel = np.log(relative_volume.replace(0, np.nan))
    rv_mean = log_rel.rolling(vol_window, min_periods=max(3, vol_window // 4)).mean()
    rv_std = log_rel.rolling(vol_window, min_periods=max(3, vol_window // 4)).std()
    volume_z = (log_rel - rv_mean) / rv_std.replace(0, np.nan)

    volume_z = volume_z.fillna(0.0)

    scale = pd.Series(MID_VOL_MULT, index=df.index)
    scale = scale.mask(volume_z <= VOL_LOW_Z, LOW_VOL_MULT)
    scale = scale.mask(volume_z >= VOL_HIGH_Z, HIGH_VOL_MULT)
    scale = scale.mask(volume_z >= VOL_BLOCK_Z, 0.0)

    # New entries are blocked during extreme volume spikes.  Existing trades
    # are merely reduced by the exposure multiplier instead of being forcibly
    # flattened immediately.
    entry_ok = volume_z < VOL_BLOCK_Z

    out["volume_z"] = volume_z
    out["volume_scale"] = scale
    out["volume_entry_ok"] = entry_ok
    return out


def build_bollinger_positions(spread, window, band_k, min_persist=2, cooldown_bars=0, allow_entry=None):
    mid = spread.rolling(window).mean()
    std = spread.rolling(window).std()
    upper = mid + band_k * std
    lower = mid - band_k * std

    position = 0
    cooldown = 0
    below_count = 0
    above_count = 0
    positions = []

    for i, (s, m, u, l) in enumerate(zip(spread, mid, upper, lower)):
        if pd.isna(s) or pd.isna(m) or pd.isna(u) or pd.isna(l):
            positions.append(0)
            continue

        below_count = below_count + 1 if s < l else 0
        above_count = above_count + 1 if s > u else 0

        if cooldown > 0:
            cooldown -= 1

        entry_ok = True if allow_entry is None else bool(allow_entry.iloc[i])

        if position == 0:
            if cooldown == 0 and entry_ok and below_count >= min_persist:
                position = 1
            elif cooldown == 0 and entry_ok and above_count >= min_persist:
                position = -1
        elif position == 1 and s >= m:
            position = 0
            cooldown = cooldown_bars
        elif position == -1 and s <= m:
            position = 0
            cooldown = cooldown_bars

        positions.append(position)

    return pd.Series(positions, index=spread.index), mid, upper, lower


def backtest_pair_strategy(
    prices, asset_a, asset_b, alpha, beta,
    strategy="zscore", window=25, entry_z=2.0, exit_z=0.25, band_k=1.0,
    fee_bps=20.0, min_persist=2, cooldown_bars=0, cost_buffer=1.0,
    bar_hours=4
):
    df = align_pair_prices(prices, asset_a, asset_b)
    df = apply_fixed_spread(df, asset_a, asset_b, alpha, beta)
    vol_info = volume_regime(df, asset_a, asset_b, bar_hours)
    df = pd.concat([df, vol_info], axis=1)

    if strategy == "zscore":
        z = rolling_zscore(df["spread"], window=window)
        expected_move = z.abs() * df["spread"].rolling(window).std()
        round_trip_cost = 2.0 * fee_bps / 10000.0
        trade_ok = expected_move > cost_buffer * round_trip_cost

        signal_position = build_zscore_positions(
            z,
            entry_z=entry_z,
            exit_z=exit_z,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars,
            allow_entry=df["volume_entry_ok"] & trade_ok,
        )
        df["zscore"] = z
        df["position_signal"] = signal_position

    elif strategy == "bollinger":
        signal_position, mid, upper, lower = build_bollinger_positions(
            df["spread"],
            window=window,
            band_k=band_k,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars,
            allow_entry=df["volume_entry_ok"],
        )
        spread_dist = (df["spread"] - mid).abs()
        round_trip_cost = 2.0 * fee_bps / 10000.0
        trade_ok = spread_dist > cost_buffer * round_trip_cost
        signal_position = signal_position.where(trade_ok, 0.0)

        df["position_signal"] = signal_position
        df["bb_mid"] = mid
        df["bb_upper"] = upper
        df["bb_lower"] = lower

    else:
        raise ValueError("strategy must be 'zscore' or 'bollinger'")

    # Low volume: modestly increase exposure.
    # High volume: reduce exposure.
    # Extreme high volume: no exposure / no new entries.
    df["position"] = df["position_signal"] * df["volume_scale"]

    gross = 1.0 + abs(beta)
    df["w_a"] = df["position"] / gross
    df["w_b"] = -beta * df["position"] / gross

    df[f"ret_{asset_a}"] = df[asset_a].pct_change()
    df[f"ret_{asset_b}"] = df[asset_b].pct_change()

    raw_ret = (
        df["w_a"].shift(1) * df[f"ret_{asset_a}"] +
        df["w_b"].shift(1) * df[f"ret_{asset_b}"]
    )

    turnover = df["w_a"].diff().abs().fillna(0) + df["w_b"].diff().abs().fillna(0)
    df["turnover"] = turnover

    cost_per_turn = fee_bps / 10000.0
    df["strategy_ret"] = raw_ret - turnover * cost_per_turn

    df = df.set_index("timestamp")
    df["equity"] = (1.0 + df["strategy_ret"].fillna(0.0)).cumprod()

    r = df["strategy_ret"].dropna()
    ann = np.sqrt((24.0 * 365.0) / bar_hours)
    vol = r.std()
    sharpe = np.nan if vol == 0 or np.isnan(vol) else (r.mean() / vol) * ann
    max_dd = (df["equity"] / df["equity"].cummax() - 1.0).min()
    cum_ret = df["equity"].iloc[-1] - 1.0 if len(df) else np.nan
    trades = int((df["position_signal"].diff().fillna(0) != 0).sum())

    metrics = pd.Series({
        "Pair": f"{asset_a}/{asset_b}",
        "Strategy": strategy,
        "Alpha": alpha,
        "Beta": beta,
        "Sharpe": sharpe,
        "Cumulative Return": cum_ret,
        "Max Drawdown": max_dd,
        "Trades": trades,
        "Avg Turnover/Bar": turnover.mean(),
        "Mean Return": r.mean(),
        "Vol": r.std(),
    })
    return df, metrics


# -----------------------------
# 4. Train/test execution helpers
# -----------------------------

def refit_selected_pairs(df, selected_pairs, bar_hours):
    """Refit alpha/beta on a larger training sample after pair identities are chosen."""
    rows = []
    for _, row in selected_pairs.iterrows():
        a, b = row["Asset_A"], row["Asset_B"]
        pair_df, alpha, beta, corr, eg_p = estimate_ols_spread(df, a, b)
        rows.append({
            **row.to_dict(),
            "Alpha": alpha,
            "Beta": beta,
            "Corr": corr,
            "EG_Coint_Pvalue": eg_p,
        })
    return pd.DataFrame(rows)


def run_pairs(prices_slice, pair_table, strategy, param_value, fee_bps, bar_hours, window_hours,
              min_persist=MIN_PERSIST, cooldown_hours=COOLDOWN_BARS_HOURS,
              cost_buffer=1.0):
    bts = {}
    ret_series = {}
    metrics_rows = []
    window = bars_from_hours(window_hours, bar_hours, minimum=5)
    cooldown_bars = bars_from_hours(cooldown_hours, bar_hours, minimum=0)

    for _, row in pair_table.iterrows():
        a, b = row["Asset_A"], row["Asset_B"]
        if strategy == "zscore":
            kwargs = dict(entry_z=param_value, exit_z=Z_EXIT)
        else:
            kwargs = dict(band_k=param_value)

        bt, m = backtest_pair_strategy(
            prices_slice,
            a,
            b,
            alpha=row["Alpha"],
            beta=row["Beta"],
            strategy=strategy,
            window=window,
            fee_bps=fee_bps,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars,
            cost_buffer=cost_buffer,
            bar_hours=bar_hours,
            **kwargs,
        )

        pair_name = f"{a}/{b}"
        bts[pair_name] = bt
        ret_series[pair_name] = bt["strategy_ret"].rename(pair_name)
        metrics_rows.append(m)

    return bts, ret_series, pd.DataFrame(metrics_rows)


def quality_adjusted_weights(ret_series, pair_metrics, max_weight=0.40):
    """Inverse-vol weights adjusted for net Sharpe, turnover, and correlation.

    All inputs are from the training sample.  Higher Sharpe helps, while higher
    turnover and higher correlation to other selected spreads reduce weight.
    """
    if not ret_series:
        return pd.Series(dtype=float), pd.DataFrame()

    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    if rets.empty:
        return pd.Series(dtype=float), pd.DataFrame()

    vols = rets.std().replace(0, np.nan)
    inv_vol = 1.0 / vols

    pm = pair_metrics.set_index("Pair").reindex(rets.columns).copy()
    corr_matrix = rets.corr().abs()
    avg_abs_corr = corr_matrix.apply(lambda s: (s.sum() - 1.0) / max(len(s) - 1, 1), axis=1)

    pm["AvgAbsPairCorr"] = avg_abs_corr
    median_turnover = pm["Avg Turnover/Bar"].median()
    if not np.isfinite(median_turnover) or median_turnover <= 0:
        median_turnover = 1.0

    sharpe_score = pm["Sharpe"].clip(lower=0).fillna(0) + 0.25
    turnover_score = 1.0 / (1.0 + pm["Avg Turnover/Bar"].fillna(median_turnover) / median_turnover)
    corr_score = 1.0 / (1.0 + avg_abs_corr.fillna(0))

    raw = inv_vol * np.sqrt(sharpe_score) * np.sqrt(turnover_score) * np.sqrt(corr_score)
    raw = raw.replace([np.inf, -np.inf], np.nan).fillna(0)

    if raw.sum() <= 0:
        raw = inv_vol.replace([np.inf, -np.inf], np.nan).fillna(0)

    weights = raw / raw.sum()

    # Simple iterative cap so one spread cannot dominate the portfolio.
    for _ in range(10):
        over = weights > max_weight
        if not over.any():
            break
        excess = (weights[over] - max_weight).sum()
        weights[over] = max_weight
        under = ~over
        if under.any() and weights[under].sum() > 0:
            weights[under] += excess * weights[under] / weights[under].sum()
        else:
            break
        weights = weights / weights.sum()

    return weights, pm


def apply_fixed_weights(ret_series, weights):
    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    w = weights.reindex(rets.columns).fillna(0.0)
    if w.sum() > 0:
        w = w / w.sum()
    port_ret = rets.mul(w, axis=1).sum(axis=1)
    return port_ret, w, rets


def portfolio_metrics(port_ret, bar_hours):
    port_ret = port_ret.dropna()
    equity = (1.0 + port_ret.fillna(0.0)).cumprod()
    vol = port_ret.std()
    ann = np.sqrt((24.0 * 365.0) / bar_hours)
    sharpe = np.nan if vol == 0 or np.isnan(vol) else (port_ret.mean() / vol) * ann
    max_dd = (equity / equity.cummax() - 1.0).min()
    return {
        "Sharpe": sharpe,
        "Cumulative Return": equity.iloc[-1] - 1.0 if len(equity) else np.nan,
        "Max Drawdown": max_dd,
        "N Obs": len(port_ret),
    }


def make_window_grid_hours(selected_pairs, bar_hours):
    """Grid = fixed practical windows + windows anchored to the selected pairs' half-lives."""
    median_hl = float(selected_pairs["HalfLifeHours"].median())
    hl_candidates = [0.5, 0.75, 1.0, 1.5, 2.0]
    candidates = [24, 48, 72, 96, 120, 168, 240, 336, 480]
    candidates += [median_hl * x for x in hl_candidates]

    normalized = []
    for h in candidates:
        h = max(2.0 * bar_hours, min(float(h), MAX_SIGNAL_WINDOW_HOURS))
        bars = bars_from_hours(h, bar_hours, minimum=5)
        eff_h = bars * bar_hours
        normalized.append(eff_h)

    return sorted(set(normalized))


def grid_search_for_timeframe(tf_hours, strategy):
    fit = DATASETS[tf_hours]["fit"]
    val = DATASETS[tf_hours]["val"]

    selected_pairs, all_pair_features = select_pairs_from_training(
        fit, bar_hours=tf_hours, fee_bps=FEE_BPS_MAIN
    )

    window_grid_hours = make_window_grid_hours(selected_pairs, tf_hours)
    print(f"\n{strategy} / {tf_hours}h: {len(selected_pairs)} pairs, window grid={window_grid_hours}")

    rows = []
    for window_hours in window_grid_hours:
        # Train-derived weights for THIS window.  Validation never determines weights.
        train_bts, train_rets, train_metrics = run_pairs(
            fit,
            selected_pairs,
            strategy,
            Z_ENTRY if strategy == "zscore" else BB_K,
            FEE_BPS_MAIN,
            bar_hours=tf_hours,
            window_hours=window_hours,
        )
        train_weights, weight_detail = quality_adjusted_weights(train_rets, train_metrics)

        val_bts, val_rets, val_metrics = run_pairs(
            val,
            selected_pairs,
            strategy,
            Z_ENTRY if strategy == "zscore" else BB_K,
            FEE_BPS_MAIN,
            bar_hours=tf_hours,
            window_hours=window_hours,
        )
        val_port, _, _ = apply_fixed_weights(val_rets, train_weights)
        metrics = portfolio_metrics(val_port, tf_hours)

        rows.append({
            "Strategy": strategy,
            "BarHours": tf_hours,
            "WindowHours": window_hours,
            "NPairs": len(selected_pairs),
            **metrics,
            "TrainMeanPairSharpe": train_metrics["Sharpe"].mean() if not train_metrics.empty else np.nan,
            "TrainMeanTurnover": train_metrics["Avg Turnover/Bar"].mean() if not train_metrics.empty else np.nan,
        })

    grid = pd.DataFrame(rows).sort_values("Sharpe", ascending=False).reset_index(drop=True)
    return grid, selected_pairs, all_pair_features


# -----------------------------
# 5. Run the actual grid search
# -----------------------------

all_grid_results = []
search_objects = {}

for strategy in ["zscore", "bollinger"]:
    for tf_hours in TIMEFRAME_HOURS:
        grid, selected_pairs, all_pair_features = grid_search_for_timeframe(tf_hours, strategy)
        all_grid_results.append(grid)
        search_objects[(strategy, tf_hours)] = {
            "selected_pairs": selected_pairs,
            "all_pair_features": all_pair_features,
        }

all_grid_results = pd.concat(all_grid_results, ignore_index=True)

print("\nValidation grid search, ranked by validation Sharpe:")
print(all_grid_results.sort_values(["Strategy", "Sharpe"], ascending=[True, False]).to_string(index=False))

best_configs = (
    all_grid_results.sort_values("Sharpe", ascending=False)
    .groupby("Strategy", as_index=False)
    .first()
)
print("\nBest configuration for each strategy:")
print(best_configs.to_string(index=False))


# -----------------------------
# 6. Refit on the FULL 70% training period and evaluate the untouched test
# -----------------------------

final_results = {}
final_weight_details = {}
final_pair_tables = {}

for _, best in best_configs.iterrows():
    strategy = best["Strategy"]
    tf_hours = int(best["BarHours"])
    window_hours = float(best["WindowHours"])

    full_train = pd.concat([DATASETS[tf_hours]["fit"], DATASETS[tf_hours]["val"]], ignore_index=True)
    test = DATASETS[tf_hours]["test"]

    # Pair identities came from the inner fit period.  Here we only REFIT the
    # hedge ratios on all pre-test observations; we do not reselect pairs using test data.
    initial_pairs = search_objects[(strategy, tf_hours)]["selected_pairs"]
    refit_pairs = refit_selected_pairs(full_train, initial_pairs, tf_hours)

    # Training returns at the selected window are used for the final weights.
    train_bts, train_rets, train_metrics = run_pairs(
        full_train,
        refit_pairs,
        strategy,
        Z_ENTRY if strategy == "zscore" else BB_K,
        FEE_BPS_MAIN,
        bar_hours=tf_hours,
        window_hours=window_hours,
    )
    final_weights, weight_detail = quality_adjusted_weights(train_rets, train_metrics)

    # The TEST uses fixed train-derived hedge ratios, fixed train-derived weights,
    # and fixed hyperparameters chosen from validation.
    test_bts, test_rets, test_metrics = run_pairs(
        test,
        refit_pairs,
        strategy,
        Z_ENTRY if strategy == "zscore" else BB_K,
        FEE_BPS_MAIN,
        bar_hours=tf_hours,
        window_hours=window_hours,
    )
    test_port, applied_weights, _ = apply_fixed_weights(test_rets, final_weights)
    metrics = portfolio_metrics(test_port, tf_hours)

    final_results[strategy] = {
        **metrics,
        "BarHours": tf_hours,
        "WindowHours": window_hours,
        "NPairs": len(refit_pairs),
        "TestReturns": test_port,
        "TestBacktests": test_bts,
        "TestPairMetrics": test_metrics,
        "Weights": applied_weights,
    }
    final_weight_details[strategy] = weight_detail
    final_pair_tables[strategy] = refit_pairs


final_summary = pd.DataFrame({
    strategy: {
        k: v for k, v in result.items()
        if k not in {"TestReturns", "TestBacktests", "TestPairMetrics", "Weights"}
    }
    for strategy, result in final_results.items()
}).T

print("\nFINAL OUT-OF-SAMPLE RESULTS")
print(final_summary.to_string())


# -----------------------------
# 7. Diagnostics: show what the model actually selected
# -----------------------------

for strategy in ["zscore", "bollinger"]:
    print(f"\n===== {strategy.upper()} PAIRS =====")
    print(
        final_pair_tables[strategy][[
            "Pair", "Corr", "EG_Coint_Pvalue", "HalfLifeHours", "BetaCV",
            "ConvergencePer100d", "ScreenSharpe", "ScreenTurnover", "SelectionScore"
        ]].to_string(index=False)
    )

    print(f"\n{strategy.upper()} FINAL WEIGHTS")
    weight_detail = final_weight_details[strategy].copy()
    weight_detail = weight_detail.join(
        final_pair_tables[strategy].set_index("Pair")[["SpreadVol"]],
        how="left"
    )
    print(
        weight_detail[[
            "Sharpe", "Avg Turnover/Bar", "AvgAbsPairCorr", "SpreadVol"
        ]].assign(Weight=final_results[strategy]["Weights"]).sort_values("Weight", ascending=False).to_string()
    )


# -----------------------------
# 8. Baseline comparison: equal-volatility weights vs quality-adjusted weights
# -----------------------------

def equal_vol_weights_from_returns(ret_series):
    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    vols = rets.std().replace(0, np.nan)
    inv_vol = 1.0 / vols
    weights = inv_vol / inv_vol.sum()
    return weights, rets

weight_compare_rows = []
for _, best in best_configs.iterrows():
    strategy = best["Strategy"]
    tf_hours = int(best["BarHours"])
    window_hours = float(best["WindowHours"])

    full_train = pd.concat([DATASETS[tf_hours]["fit"], DATASETS[tf_hours]["val"]], ignore_index=True)
    test = DATASETS[tf_hours]["test"]
    refit_pairs = final_pair_tables[strategy]

    _, train_rets, train_metrics = run_pairs(
        full_train,
        refit_pairs,
        strategy,
        Z_ENTRY if strategy == "zscore" else BB_K,
        FEE_BPS_MAIN,
        bar_hours=tf_hours,
        window_hours=window_hours,
    )

    eq_weights, _ = equal_vol_weights_from_returns(train_rets)
    qa_weights = final_results[strategy]["Weights"]

    _, test_rets, _ = run_pairs(
        test,
        refit_pairs,
        strategy,
        Z_ENTRY if strategy == "zscore" else BB_K,
        FEE_BPS_MAIN,
        bar_hours=tf_hours,
        window_hours=window_hours,
    )

    eq_port, _, _ = apply_fixed_weights(test_rets, eq_weights)
    qa_port, _, _ = apply_fixed_weights(test_rets, qa_weights)

    row = {"Strategy": strategy}
    row.update({f"EqualVol_{k}": v for k, v in portfolio_metrics(eq_port, tf_hours).items()})
    row.update({f"QualityAdj_{k}": v for k, v in portfolio_metrics(qa_port, tf_hours).items()})
    weight_compare_rows.append(row)

weight_comparison = pd.DataFrame(weight_compare_rows)
print("\nWeighting comparison on the untouched test period:")
print(weight_comparison.to_string(index=False))


# -----------------------------
# 9. Test-period equity curves
# -----------------------------

plt.figure(figsize=(11, 5))
for strategy in ["zscore", "bollinger"]:
    ret = final_results[strategy]["TestReturns"]
    plt.plot((1.0 + ret.fillna(0.0)).cumprod(), label=strategy)
plt.title("Out-of-Sample Equity Curves")
plt.xlabel("Time")
plt.ylabel("Equity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# -----------------------------
# 10. Optional: cost sensitivity on the FINAL selected configurations
# -----------------------------

fee_grid = [0, 5, 10, 15, 20, 25]
cost_rows = []

for _, best in best_configs.iterrows():
    strategy = best["Strategy"]
    tf_hours = int(best["BarHours"])
    window_hours = float(best["WindowHours"])
    test = DATASETS[tf_hours]["test"]
    full_train = pd.concat([DATASETS[tf_hours]["fit"], DATASETS[tf_hours]["val"]], ignore_index=True)
    refit_pairs = final_pair_tables[strategy]

    for fee in fee_grid:
        _, train_rets, train_metrics = run_pairs(
            full_train, refit_pairs, strategy,
            Z_ENTRY if strategy == "zscore" else BB_K,
            fee,
            bar_hours=tf_hours,
            window_hours=window_hours,
        )
        train_weights, _ = quality_adjusted_weights(train_rets, train_metrics)

        _, test_rets, _ = run_pairs(
            test, refit_pairs, strategy,
            Z_ENTRY if strategy == "zscore" else BB_K,
            fee,
            bar_hours=tf_hours,
            window_hours=window_hours,
        )
        port_ret, _, _ = apply_fixed_weights(test_rets, train_weights)
        row = {"Strategy": strategy, "FeeBps": fee}
        row.update(portfolio_metrics(port_ret, tf_hours))
        cost_rows.append(row)

cost_df = pd.DataFrame(cost_rows)
print("\nCost sensitivity:")
print(cost_df.to_string(index=False))

plt.figure(figsize=(10, 5))
for strategy in ["zscore", "bollinger"]:
    sub = cost_df[cost_df["Strategy"] == strategy].sort_values("FeeBps")
    plt.plot(sub["FeeBps"], sub["Sharpe"], marker="o", label=strategy)
plt.title("Out-of-Sample Cost Sensitivity")
plt.xlabel("Transaction Cost (bps)")
plt.ylabel("Sharpe")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ===================== END REPLACEMENT BLOCK =====================


In [ ]:
# i need to check 4h, 8h, 12h resampling

# i need to get more specific on the rolling signal window (i just used an estimate for now). what constitutes a half-life that is "too long"
# is the mean half-life ok? intuition, what others have found, usually <20 days shows reversal, too far beyond a month is like momentum
# check crypto vs equities
# gridsearch, cross validate (val set)

# Since this is a reversal strategy, I would test a low-volume filter. Deviations that occur on low volume may be more 
# likely to reflect temporary liquidity imbalances, while high-volume moves may reflect new information and continue trending. 
# You could increase exposure during lower-volume periods and reduce or avoid trades during unusually high-volume periods.

# Equal-volatility weighting is a reasonable baseline. As a further improvement, you could use training-period 
# estimates of pair correlation, turnover and net Sharpe to reduce the weight of expensive or highly correlated spreads.

# Strengthen the pair-selection criteria using training-period information. In addition to cointegration, 
# consider the stability of the hedge ratio, estimated half-life, spread volatility, frequency of convergence and n
# et performance after estimated costs. A pair can be statistically cointegrated but still be too slow or expensive to trade profitably.

# Consider rolling pair selection and hedge-ratio estimation. Crypto relationships can change over time, 
# so you could re-estimate eligible pairs quarterly using only trailing data and then trade those pairs during 
# the following quarter. This would remain point-in-time and may help when relationships weaken.

# Although you begin with a 20-coin universe, only two pairs make it into the final portfolio. That leaves the strategy 
# with limited diversification and several concentrated coin exposures. You could expand the initial universe or 
# explore cointegrated baskets, but I would not loosen the statistical criteria solely to produce more pairs.
 

